# 🔬 govAI — Calibração da camada epistemológica

**Objetivo:** verificar, antes da anotação completa, se a postura **DN** é identificável
por dois anotadores independentes — medindo α por dimensão, com foco em **α_DN**.

Tipologia: ternária multi-rótulo com três marcadores positivos e independentes
(`epi_positivista`, `epi_interpretativa`, `epi_doutrinario_normativa`). Qualquer
combinação é válida; três zeros não derivam DN (resultam em `inconclusivo`).

**Não é** o piloto formal de decisão tipológica (arquivado): a tipologia já foi decidida
em bases conceituais (Codebook DA-08). Aqui é só a verificação de viabilidade.

**Como usar:** clique ▶ em cada célula, de cima para baixo. Espere ✓ antes da próxima.

> ⚠️ O passo de embeddings (opcional, exploratório) usa GPU: `Runtime → Change runtime type → T4 GPU`.

---
## BLOCO 0 — Configuração (rode uma vez por sessão)


In [ ]:
# 0-A: Montar o Google Drive (seus arquivos ficam salvos entre sessões)
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/govai_pilot'
os.makedirs(BASE, exist_ok=True)
print('Drive montado. Pasta do projeto:', BASE)


In [ ]:
# 0-B: Carregar o código do projeto a partir do zip
# Faça upload do arquivo govai_pilot_code.zip (baixado da sessão de trabalho)
from google.colab import files
import zipfile, shutil

print('Selecione govai_pilot_code.zip quando a caixa abrir abaixo:')
uploaded = files.upload()

zipfile.ZipFile('govai_pilot_code.zip').extractall('.')
print('Código extraído. Conteúdo da pasta pilot:',
      [f for f in os.listdir('pilot')])


In [ ]:
# 0-C: Instalar dependências (≈ 2-3 minutos na primeira vez)
!pip install pandas scikit-learn krippendorff requests transformers torch -q
print('Instalação concluída.')


---
## BLOCO 1 — Selecionar os artigos para o piloto

Você precisa de um arquivo **`corpus.csv`** com pelo menos as colunas:
- `doc_id` — identificador único de cada artigo
- colunas de estrato: ex. `cluster`, `area` (PA / IS), `region` (BR / intl)

Se ainda não tem o corpus em CSV, exporte de Notion / Zotero / Scopus com essas colunas.
O script vai sortear **150 artigos representativos** + **50 artigos de reforço de DN** (cluster Law).


In [ ]:
# 1-A: Fazer upload do corpus.csv
print('Selecione corpus.csv:')
uploaded = files.upload()
import shutil
shutil.copy('corpus.csv', f'{BASE}/corpus.csv')
print('corpus.csv salvo no Drive.')


In [ ]:
# 1-B: Sortear a amostra de calibração (~25 artigos, estratificada, sem reforço de DN)
# Inline: não depende de pilot/sample_selection.py. Estratifica por cluster_origem.
import pandas as pd

N_CALIB = 25          # alvo de 20–30; ajuste se quiser
SEED = 42
corpus = pd.read_csv('corpus.csv')

strat_col = 'cluster_origem' if 'cluster_origem' in corpus.columns else None
if strat_col:
    frac = N_CALIB / len(corpus)
    sample = (corpus.groupby(strat_col, group_keys=False)
                    .apply(lambda g: g.sample(max(1, round(len(g)*frac)), random_state=SEED)))
    sample = sample.sample(min(N_CALIB, len(sample)), random_state=SEED)
else:
    sample = corpus.sample(N_CALIB, random_state=SEED)

sample = sample[['doc_id']].copy()
sample['subsample'] = 'calibracao'
sample.to_csv('sample_calib.csv', index=False)
print(f'Amostra de calibração: {len(sample)} artigos')
shutil.copy('sample_calib.csv', f'{BASE}/sample_calib.csv')
print('sample_calib.csv salvo no Drive.')

# Se após a 1ª rodada houver < ~5 casos DN positivos, acrescente 5–10 artigos de
# periódicos jurídicos (subpopulacao_juridica==1), analisados à parte (ver Protocolo de Calibração §2).

---
## BLOCO 2 — Baixar os abstracts

**Modo A (recomendado se você tem os DOIs):** prepare um `dois.csv` com coluna `doi`
e opcionalmente `doc_id`. Execute a célula 2-A.

**Modo B (filtro por periódico no OpenAlex):** edite o filtro na célula 2-B.

O resultado é `abstracts.csv` (doc_id, abstract).


In [ ]:
# 2-A: Por lista de DOIs  ← execute esta OU a 2-B, não as duas
print('Selecione dois.csv (colunas: doi e, opcionalmente, doc_id):')
uploaded = files.upload()
!python pilot/download_abstracts.py --dois dois.csv --mailto voce@fgv.br
shutil.copy('abstracts.csv', f'{BASE}/abstracts.csv')
import pandas as pd; df=pd.read_csv('abstracts.csv')
print(f'{len(df)} abstracts baixados. Primeiras linhas:'); print(df.head(2))


In [ ]:
# 2-B: Por filtro OpenAlex  ← execute esta OU a 2-A, não as duas
# Troque o filtro pelo periódico e período do seu corpus.
# Para achar o source.id: acesse https://api.openalex.org/sources?search=NOME_DO_PERIODICO
FILTRO = 'primary_location.source.id:S137773608,from_publication_date:2015-01-01'
!python pilot/download_abstracts.py --filter '{FILTRO}' --mailto voce@fgv.br
shutil.copy('abstracts.csv', f'{BASE}/abstracts.csv')
import pandas as pd; df=pd.read_csv('abstracts.csv')
print(f'{len(df)} abstracts baixados.'); print(df.head(2))


---
## BLOCO 3 — [OPCIONAL] Pré-classificação exploratória por LLM

Este bloco é uma **triagem provisória** — não substitui a anotação humana.
Serve para ter uma leitura rápida antes de anotar manualmente.

Para esta triagem exploratória no Colab usa-se a API da Anthropic por
conveniência. **No pipeline de produção, a pré-classificação canônica é o
script `04b_classificar_epi_llm.py`, que usa OpenRouter** (Manual §1.8, §4.2bis).
Os resultados deste bloco são rotulados como exploratórios e não entram no
Gold Standard.

Para rodar, você precisa de uma **chave da API da Anthropic**.
Entre em https://console.anthropic.com → API Keys → copie a chave.

In [ ]:
# 3: Pré-classificação por LLM (triagem exploratória — NÃO é padrão-ouro)
from google.colab import userdata
import os, shutil

# Secrets (ícone 🔑 no painel esquerdo) → Add new secret → ANTHROPIC_API_KEY
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

# NOTA: o prompt deve pedir TRÊS flags positivos e independentes —
#   epi_positivista, epi_interpretativa, epi_doutrinario_normativa —
# não mais duas dimensões + derivação por exclusão (EPI_NA está deprecado).
# A versão canônica do prompt está em 04b_classificar_epi_llm.py.
!python pilot/llm_prepass.py abstracts.csv
shutil.copy('annotations_llm_prepass.csv', f'{BASE}/annotations_llm_prepass.csv')
print('Pronto. Resultado salvo no Drive (rotulado como exploratório).')

---
## ⏸️ PAUSA — Anotação humana

Agora você e o(a) segundo(a) anotador(a) anotam os artigos de forma
**independente** (sem comparar respostas), seguindo o
**Guia de Anotação de Posturas v4 (3_Guia_Anotacao_Posturas_v4.docx)**.

**Esquema ternário multi-rótulo (Addendum 1 ao OSF):** três flags positivos
e independentes, cada um 0 ou 1. Não há derivação por exclusão.

**Resultado esperado:** dois arquivos `calib_ann1.csv` e `calib_ann2.csv`,
uma linha por `doc_id`, com as colunas canônicas:
```
doc_id, epi_positivista, epi_interpretativa, epi_doutrinario_normativa
```
(cada coluna 0/1; opcionalmente `confianca` 0–3 e `notas`).

Os nomes das colunas devem ser exatamente esses — os mesmos de
`anotacao_calibracao.csv` e do Gold Standard. Quando as anotações
estiverem prontas, continue no Bloco 4.

---
## BLOCO 4 — Análise dos rótulos e concordância inter-anotador


In [ ]:
# 4: Carregar calib_ann1.csv + calib_ann2.csv e calcular α por dimensão
# Colunas esperadas (canônicas, iguais ao Gold Standard):
#   doc_id, epi_positivista, epi_interpretativa, epi_doutrinario_normativa (0/1)
#   opcionais: confianca (0-3), notas (com dn:modo/dn:norm/dn:ambos quando DN=1)
import pandas as pd, numpy as np, os, sys
try:
    import krippendorff
except ImportError:
    !pip install krippendorff -q
    import krippendorff

# Derivação determinística canônica (utils/derive_orientacao.py)
for _p in [".", "utils", "pilot", "pilot/utils"]:
    if os.path.isfile(os.path.join(_p, "derive_orientacao.py")) and _p not in sys.path:
        sys.path.insert(0, _p)
try:
    from derive_orientacao import derive, validar_linha
    _DERIVE = True
except Exception:
    _DERIVE = False
    print("[aviso] derive_orientacao.py não encontrado; QA determinístico e proeminente desativados.")

FLAGS = ['epi_positivista', 'epi_interpretativa', 'epi_doutrinario_normativa']

print('Selecione calib_ann1.csv e calib_ann2.csv:')
up = files.upload()
a1 = pd.read_csv('calib_ann1.csv').set_index('doc_id').sort_index()
a2 = pd.read_csv('calib_ann2.csv').set_index('doc_id').sort_index()

# Validação de schema: falhar alto se as colunas canônicas não existirem.
for nome, dfa in [('calib_ann1', a1), ('calib_ann2', a2)]:
    faltam = [c for c in FLAGS if c not in dfa.columns]
    if faltam:
        raise KeyError(f'{nome}.csv sem colunas {faltam}. '
                       f'Use os nomes canônicos: {FLAGS}. Presentes: {list(dfa.columns)}')

docs = a1.index.intersection(a2.index)
a1, a2 = a1.loc[docs], a2.loc[docs]

def alpha_bin(col):
    # reliability_data: linhas = anotadores, colunas = unidades; nominal binário
    data = np.vstack([a1[col].values, a2[col].values])
    return krippendorff.alpha(reliability_data=data, level_of_measurement='nominal')

print('=== α de Krippendorff por dimensão (exploratório; α canônico = irrCAC/R) ===')
rotulos = [('epi_doutrinario_normativa', 'α_DN  (piso calib. ≥ 0,667)'),
           ('epi_positivista',           'α_positivista'),
           ('epi_interpretativa',        'α_interpretativa')]
for col, nome in rotulos:
    try:    print(f'  {nome:22s}: {alpha_bin(col): .3f}')
    except Exception as e: print(f'  {nome}: erro ({e})')

# Piso de viabilidade da calibração de DN (Addendum 2): α_DN ≥ 0,667
try:
    a_dn = alpha_bin('epi_doutrinario_normativa')
    print(f'\n[GATE] α_DN = {a_dn:.3f} — '
          + ('VIÁVEL (≥ 0,667)' if a_dn >= 0.667 else 'ABAIXO do piso (revisar protocolo/calibração)'))
except Exception as e:
    print(f'[GATE] não foi possível avaliar α_DN: {e}')

# Distribuição de padrões multi-rótulo (onde ambos concordam)
both = (a1[FLAGS].astype(int) & a2[FLAGS].astype(int))
sig = ['pos','int','dn']
pat = both.apply(lambda r: '+'.join(s for s,v in zip(sig, r) if v) or 'inconclusivo', axis=1)
print('\n=== Distribuição de padrões (onde ambos concordam) ===')
print(pat.value_counts().to_string())
print(f'\nTaxa inconclusivo (três zeros): {(pat=="inconclusivo").mean():.1%}')
# mixed canônico = pos AND int (exclusivamente)
mixed = ((both['epi_positivista']==1) & (both['epi_interpretativa']==1))
print(f'Taxa mixed (pos AND int): {mixed.mean():.1%}')

# === Eixo 1: orientacao_proeminente derivada das flags, e concordância no rótulo ===
if _DERIVE:
    prom1 = a1.apply(lambda r: derive(r['epi_positivista'], r['epi_interpretativa'], r['epi_doutrinario_normativa'])['orientacao_proeminente'], axis=1)
    prom2 = a2.apply(lambda r: derive(r['epi_positivista'], r['epi_interpretativa'], r['epi_doutrinario_normativa'])['orientacao_proeminente'], axis=1)
    print('\n=== Eixo 1 (orientacao_proeminente derivada) ===')
    print(f'Concordância no rótulo do eixo 1: {(prom1 == prom2).mean():.1%}')
    print('ann1:', prom1.value_counts().to_dict())
    print('ann2:', prom2.value_counts().to_dict())

# === QA determinístico das anotações (Guia V4: confianca 0-3, notas, dn:) ===
if _DERIVE and ('confianca' in a1.columns or 'confianca' in a2.columns):
    for nome, dfa in [('calib_ann1', a1), ('calib_ann2', a2)]:
        if 'confianca' not in dfa.columns:
            continue
        n_prob = 0
        for doc, r in dfa.iterrows():
            v = validar_linha(r['epi_positivista'], r['epi_interpretativa'],
                              r['epi_doutrinario_normativa'], r.get('confianca'), r.get('notas'))
            if v:
                n_prob += 1
                print(f'  [{nome} {doc}] {v}')
        print(f'{nome}: {n_prob} linha(s) com problema de QA')

# === Discordâncias de DN por sub-etiqueta (decide o split de DN) ===
# DA-08: quando DN=1, a etiqueta dn:modo/dn:norm/dn:ambos é registrada em notas.
def _dn_subtag(notas):
    t = str(notas).lower()
    for tag in ('dn:ambos', 'dn:modo', 'dn:norm'):
        if tag in t:
            return tag.split(':')[1]
    return None

if ('notas' in a1.columns) and ('notas' in a2.columns):
    disc = [d for d in docs if int(a1.loc[d, 'epi_doutrinario_normativa']) != int(a2.loc[d, 'epi_doutrinario_normativa'])]
    print(f'\n=== Discordâncias em DN: {len(disc)} de {len(docs)} ===')
    if disc:
        tab = pd.DataFrame({
            'ann1_DN': [int(a1.loc[d, 'epi_doutrinario_normativa']) for d in disc],
            'ann2_DN': [int(a2.loc[d, 'epi_doutrinario_normativa']) for d in disc],
            'ann1_dn': [_dn_subtag(a1.loc[d, 'notas']) for d in disc],
            'ann2_dn': [_dn_subtag(a2.loc[d, 'notas']) for d in disc],
        }, index=disc)
        print(tab.to_string())
    sub = []
    for dfa in (a1, a2):
        m = dfa['epi_doutrinario_normativa'].astype(int) == 1
        sub.extend(dfa.loc[m, 'notas'].map(_dn_subtag).tolist())
    print('\nDistribuição de sub-etiquetas dn: (onde DN=1):')
    print(pd.Series(sub, dtype=object).value_counts(dropna=False).to_string())
    print('\nLeitura: se as discordâncias de DN se concentram numa sub-etiqueta (p.ex. dn:norm),')
    print('há evidência para desdobrar DN (modo doutrinário vs orientação normativa) antes de travar o Codebook.')
else:
    print("\n[info] sem coluna 'notas' nos dois anotadores; análise de sub-etiquetas dn: pulada.")


---
## Embeddings e separabilidade (opcional, exploratório)

Estes passos eram sondas do piloto formal. Na calibração são opcionais e apenas
exploratórios — a decisão de entrada na anotação usa α_DN (célula 4), não a separabilidade.

In [ ]:
# 5: Gerar embeddings BERTimbau para os abstracts
!python pilot/embeddings.py abstracts.csv

shutil.copy('embeddings.npy', f'{BASE}/embeddings.npy')
shutil.copy('doc_ids.csv',    f'{BASE}/doc_ids.csv')

import numpy as np
X = np.load('embeddings.npy')
print(f'Embeddings gerados: {X.shape} (artigos × dimensão)')
print('Salvos no Drive.')


---
## BLOCO 6 — Separabilidade e decisão


In [ ]:
# 6: Critério de entrada na anotação completa (lê α_DN da célula 4)
print('=== CRITÉRIO DE ENTRADA (Protocolo de Calibração §6) ===')
print('  α_DN >= 0,667 e  all-zero raro      → prosseguir para a anotação completa')
print('  α_DN <  0,667                         → refinar o guia e recalibrar')
print('  all-zero ≳ 15%                        → rever esquema (4ª categoria / código OUTRO)')
print()
print('Lembrete: 0,667 é o piso de viabilidade da calibração de DN (Addendum 2); o gate de')
print('aceitação do Gold Standard é α >= 0,55. O piso de calibração agora EXCEDE o gate')
print('(inversão deliberada: a pré-checagem de calibração é mais estrita que a aceitação).')
print('O α canônico para o registro usa irrCAC (R); o valor acima é leitura rápida exploratória.')

In [ ]:
# 6b: (removido) A sonda B1/B2 e o TF-IDF pertenciam ao piloto formal de decisão,
# agora arquivado. A calibração decide por α_DN (célula 4) + inspeção das divergências.
print('Célula sem ação nesta versão de calibração.')

---
## BLOCO 7 — Baixar os resultados para o seu Mac

Todos os arquivos já foram salvos no seu Google Drive (pasta `govai_pilot`).
Para baixar diretamente do Colab, use a célula abaixo.


In [ ]:
# 7: Baixar arquivos de resultado
from google.colab import files
for fname in ['sample_calib.csv','abstracts.csv','annotations_llm_prepass.csv',
              'embeddings.npy','doc_ids.csv']:
    try: files.download(fname)
    except: pass
print('Download iniciado para os arquivos disponíveis.')


---
## ❓ Problemas comuns

| Sintoma | Solução |
|---|---|
| `ModuleNotFoundError` | Rode novamente a célula 0-C (pip install) |
| `FileNotFoundError: corpus.csv` | Refaça o upload na célula correspondente |
| Célula demora mais de 10 min sem GPU | Ative a GPU: Runtime → Change runtime type → T4 GPU |
| `RuntimeError: CUDA out of memory` | Runtime → Disconnect and delete runtime → reconecte |
| `ANTHROPIC_API_KEY not found` | Adicione a chave em Secrets (ícone 🔑 no painel esquerdo) |
| Sessão expirou e arquivos sumiram | Os arquivos estão no Google Drive; recarregue com a célula 0-A |

---
*govAI · FAPESP Pós-doc · FGV EAESP · Fernando Leite*
